# Práctica · Semana 4: Juegos y decisiones con adversario

**Curso:** Introducción a la Inteligencia Artificial · UDES
**Tiempo estimado:** 2 h 30 min
**Relación con el capítulo:** secciones 4.1 a 4.6

En este cuaderno vas a construir agentes que juegan contra un adversario. Empezarás por el **tres en raya**, que es lo bastante pequeño para explorar su árbol de juego completo, y terminarás con **Conecta 4**, donde ya no es posible y hay que cortar la búsqueda y usar una función de evaluación.

| Parte | Juego | Técnicas |
|---|---|---|
| 1 a 4 | Tres en raya | Minimax, negamax, poda alfa-beta, ordenación de movimientos |
| 5 y 6 | Conecta 4 | Alfa-beta con profundidad limitada y funciones de evaluación |
| 7 | Un juego con dado | Expectiminimax y escala de la función de evaluación |

Ejecuta las celdas en orden. Las celdas marcadas con ✏️ son ejercicios. Todas las celdas se probaron con Python 3 y no necesitan librerías externas.

In [ ]:
import math
import random
import time

## 1. El tres en raya como problema de búsqueda

Russell y Norvig (2004, cap. 6) definen un juego con cuatro componentes, parecidos a los de un problema de búsqueda:

| Componente | En el tres en raya |
|---|---|
| **Estado inicial** | Tablero vacío; empieza MAX (`X`). |
| **Función sucesor** | Los movimientos legales: poner la ficha del jugador en turno en una casilla vacía. |
| **Test terminal** | ¿Hay tres en línea o el tablero está lleno? |
| **Función de utilidad** | +1 si gana `X`, −1 si gana `O`, 0 si es empate. |

El tablero es una tupla de 9 casillas leídas por filas: `"X"`, `"O"` o `" "`.

In [ ]:
LINEAS = [(0, 1, 2), (3, 4, 5), (6, 7, 8),      # filas
          (0, 3, 6), (1, 4, 7), (2, 5, 8),      # columnas
          (0, 4, 8), (2, 4, 6)]                 # diagonales
VACIO = (" ",) * 9


def ganador(t):
    for a, b, c in LINEAS:
        if t[a] != " " and t[a] == t[b] == t[c]:
            return t[a]
    return None


def turno(t):
    return "X" if t.count("X") == t.count("O") else "O"


def es_terminal(t):
    return ganador(t) is not None or " " not in t


def utilidad(t):
    # Desde el punto de vista de MAX (X).
    g = ganador(t)
    return 1 if g == "X" else -1 if g == "O" else 0


def movimientos(t):
    return [i for i in range(9) if t[i] == " "]


def jugar(t, i):
    return t[:i] + (turno(t),) + t[i + 1:]


def mostrar(t):
    filas = [" | ".join(t[f * 3:f * 3 + 3]) for f in range(3)]
    print("\n---------\n".join(filas))


mostrar(jugar(jugar(VACIO, 4), 0))

## 2. Minimax

MAX elige el movimiento que **maximiza** la utilidad suponiendo que MIN, en su turno, elegirá el que la **minimiza** (sección 4.3). El valor minimax de un estado se calcula recursivamente:

$$
\text{MINIMAX}(s) =
\begin{cases}
\text{UTILIDAD}(s) & \text{si } s \text{ es terminal} \\
\max_{a} \text{MINIMAX}(\text{RESULTADO}(s, a)) & \text{si juega MAX} \\
\min_{a} \text{MINIMAX}(\text{RESULTADO}(s, a)) & \text{si juega MIN}
\end{cases}
$$

El contador `nodos` registra cuántos estados visita la búsqueda.

In [ ]:
nodos = 0


def minimax(t):
    global nodos
    nodos += 1
    if es_terminal(t):
        return utilidad(t)
    valores = [minimax(jugar(t, m)) for m in movimientos(t)]
    return max(valores) if turno(t) == "X" else min(valores)


def decision_minimax(t):
    # Devuelve el mejor movimiento para el jugador en turno.
    hijos = {m: minimax(jugar(t, m)) for m in movimientos(t)}
    return (max if turno(t) == "X" else min)(hijos, key=hijos.get), hijos


nodos = 0
t0 = time.perf_counter()
valor = minimax(VACIO)
print(f"Valor minimax del tablero vacío: {valor}")
print(f"Nodos visitados: {nodos:,}  |  tiempo: {time.perf_counter() - t0:.1f} s")

El valor 0 significa que, si los dos jugadores juegan de forma perfecta, el tres en raya termina **en empate**. Para saberlo, minimax recorrió el árbol de juego completo.

### ✏️ Ejercicio 1. Leer el árbol

1. Ejecuta la celda siguiente. En la posición dada juega `O`. ¿Qué valor tiene cada movimiento posible? ¿Cuál elige minimax y por qué?
2. ¿Por qué hay movimientos con valor +1 si el que juega es `O`? ¿Qué significa ese valor?
3. El árbol completo tiene 549.946 nodos, pero el tres en raya solo tiene 5.478 estados legales distintos. ¿Por qué esa diferencia? Relaciónala con los estados repetidos del capítulo 2.

In [ ]:
posicion = ("X", " ", " ",
            " ", "O", " ",
            " ", " ", "X")
mostrar(posicion)
print("\nJuega:", turno(posicion))
mejor, valores = decision_minimax(posicion)
print("Valores por casilla:", valores)
print("Minimax elige la casilla", mejor)

*Tus respuestas al ejercicio 1:*

## 3. Negamax

En un juego de **suma cero**, lo que gana un jugador lo pierde el otro: $\text{valor}_{\text{MIN}} = -\text{valor}_{\text{MAX}}$. **Negamax** aprovecha esa simetría para escribir minimax con una sola función: cada jugador maximiza su propio valor, que es el **negativo** del valor de su rival (sección 4.3.3). El código es más corto y se usa en casi todos los programas de juegos reales.

In [ ]:
def negamax(t):
    # Valor del estado para el jugador que tiene el turno.
    global nodos
    nodos += 1
    if es_terminal(t):
        signo = 1 if turno(t) == "X" else -1
        return signo * utilidad(t)
    return max(-negamax(jugar(t, m)) for m in movimientos(t))


nodos = 0
print("Negamax del tablero vacío (para X):", negamax(VACIO), "| nodos:", f"{nodos:,}")

## 4. Poda alfa-beta

La poda **alfa-beta** obtiene **el mismo valor** que minimax sin explorar las ramas que no pueden cambiar la decisión (sección 4.5). Mantiene dos cotas:

- **α**: el mejor valor que MAX ya tiene asegurado en el camino actual.
- **β**: el mejor valor que MIN ya tiene asegurado en el camino actual.

En cuanto α ≥ β, el resto de los hijos del nodo actual se **poda**: el jugador que está más arriba nunca permitiría llegar aquí.

In [ ]:
def alfabeta(t, alfa=-math.inf, beta=math.inf, orden=None):
    global nodos
    nodos += 1
    if es_terminal(t):
        return utilidad(t)
    movs = movimientos(t) if orden is None else orden(t)
    if turno(t) == "X":                         # MAX
        v = -math.inf
        for m in movs:
            v = max(v, alfabeta(jugar(t, m), alfa, beta, orden))
            alfa = max(alfa, v)
            if alfa >= beta:
                break                           # poda beta
        return v
    else:                                       # MIN
        v = math.inf
        for m in movs:
            v = min(v, alfabeta(jugar(t, m), alfa, beta, orden))
            beta = min(beta, v)
            if alfa >= beta:
                break                           # poda alfa
        return v


nodos = 0
t0 = time.perf_counter()
v = alfabeta(VACIO)
print(f"Alfa-beta: valor {v} | nodos: {nodos:,} | tiempo: {time.perf_counter() - t0:.2f} s")

### ✏️ Ejercicio 2. El orden de los movimientos importa

La eficacia de alfa-beta depende del orden en que se examinan los hijos: si los mejores movimientos se exploran primero, se poda mucho más. Con un orden perfecto, alfa-beta examina unos $O(b^{m/2})$ nodos en lugar de $O(b^m)$ (Knuth y Moore, 1975; sección 4.5.3).

1. Ejecuta la celda. Compara los nodos visitados por minimax y por alfa-beta con tres órdenes: el natural (casillas 0 a 8), uno aleatorio y uno «inteligente» que prueba primero el centro, después las esquinas y al final los bordes.
2. ¿Cuál es el porcentaje de nodos que se ahorra con cada orden? ¿Cambia el valor del juego?
3. ¿Por qué tiene sentido probar primero el centro en el tres en raya?

In [ ]:
def orden_aleatorio(t):
    m = movimientos(t)
    random.shuffle(m)
    return m


PREFERENCIA = [4, 0, 2, 6, 8, 1, 3, 5, 7]       # centro, esquinas, bordes


def orden_inteligente(t):
    return [i for i in PREFERENCIA if t[i] == " "]


random.seed(0)
print(f"{'Algoritmo':<32}{'valor':>6}{'nodos':>12}{'% de minimax':>15}")
nodos = 0
minimax(VACIO)
total_minimax = nodos
print(f"{'Minimax':<32}{0:>6}{total_minimax:>12,}{100:>14.1f}%")
for nombre, orden in [("Alfa-beta, orden natural", None),
                      ("Alfa-beta, orden aleatorio", orden_aleatorio),
                      ("Alfa-beta, centro y esquinas", orden_inteligente)]:
    nodos = 0
    v = alfabeta(VACIO, orden=orden)
    print(f"{nombre:<32}{v:>6}{nodos:>12,}{100 * nodos / total_minimax:>14.1f}%")

*Tus respuestas al ejercicio 2:*

### 4.1 ¿Juega bien? Un torneo contra un jugador aleatorio

Un agente minimax nunca debería perder. Lo comprobamos con 200 partidas contra un jugador que elige casillas al azar, alternando quién empieza. Usamos alfa-beta con el orden inteligente para que sea rápido.

In [ ]:
def mejor_movimiento(t):
    # Mejor movimiento para el jugador en turno usando alfa-beta.
    jugador = turno(t)
    mejor, mejor_v = None, None
    for m in orden_inteligente(t):
        v = alfabeta(jugar(t, m), orden=orden_inteligente)
        if mejor_v is None or (v > mejor_v if jugador == "X" else v < mejor_v):
            mejor, mejor_v = m, v
    return mejor


def partida(jugador_x, jugador_o):
    t = VACIO
    while not es_terminal(t):
        t = jugar(t, (jugador_x if turno(t) == "X" else jugador_o)(t))
    return ganador(t) or "empate"


aleatorio = lambda t: random.choice(movimientos(t))
random.seed(1)
resultados = {"IA gana": 0, "empate": 0, "IA pierde": 0}
for k in range(200):
    if k % 2 == 0:
        r = partida(mejor_movimiento, aleatorio)
        clave = "IA gana" if r == "X" else "IA pierde" if r == "O" else "empate"
    else:
        r = partida(aleatorio, mejor_movimiento)
        clave = "IA gana" if r == "O" else "IA pierde" if r == "X" else "empate"
    resultados[clave] += 1
print(resultados)

### 4.2 Juega tú

Ejecuta la celda siguiente y escribe el número de la casilla (0 a 8, leídas por filas) cuando te lo pida. Puedes elegir si empiezas o no. Intenta ganarle.

In [ ]:
def jugar_contra_ia(empieza_humano=True):
    t = VACIO
    humano = "X" if empieza_humano else "O"
    while not es_terminal(t):
        if turno(t) == humano:
            mostrar(t)
            i = int(input(f"Tu jugada ({humano}), casilla 0-8: "))
            if i not in movimientos(t):
                print("Casilla no válida.")
                continue
        else:
            i = mejor_movimiento(t)
            print(f"La IA juega en la casilla {i}")
        t = jugar(t, i)
    mostrar(t)
    print("Resultado:", ganador(t) or "empate")

# Quita el comentario de la línea siguiente para jugar:
# jugar_contra_ia(empieza_humano=True)

## 5. Conecta 4: cuando el árbol es demasiado grande

Conecta 4 se juega en un tablero de 6 filas por 7 columnas; las fichas caen por gravedad y gana quien alinee cuatro. Tiene del orden de $4{,}5 \times 10^{12}$ posiciones legales: recorrer el árbol completo con Python no es viable.

La solución de Shannon (1950), que siguen todos los programas de juegos, es **cortar la búsqueda** a una profundidad fija y aplicar en las hojas una **función de evaluación** que estime qué tan buena es la posición (sección 4.4).

Representamos el tablero como una lista de 7 columnas, cada una con las fichas desde abajo.

In [ ]:
FILAS, COLS = 6, 7


def c4_vacio():
    return tuple(() for _ in range(COLS))


def c4_turno(t):
    return 1 if sum(len(c) for c in t) % 2 == 0 else 2


def c4_movs(t):
    return [c for c in range(COLS) if len(t[c]) < FILAS]


def c4_jugar(t, c):
    return t[:c] + (t[c] + (c4_turno(t),),) + t[c + 1:]


def c4_celda(t, f, c):
    return t[c][f] if f < len(t[c]) else 0


def c4_ventanas(t):
    # Todas las ventanas de 4 celdas: horizontales, verticales y diagonales.
    for f in range(FILAS):
        for c in range(COLS):
            for df, dc in [(0, 1), (1, 0), (1, 1), (-1, 1)]:
                if 0 <= f + 3 * df < FILAS and 0 <= c + 3 * dc < COLS:
                    yield [c4_celda(t, f + k * df, c + k * dc) for k in range(4)]


def c4_ganador(t):
    for v in c4_ventanas(t):
        if v[0] and v.count(v[0]) == 4:
            return v[0]
    return 0


def c4_terminal(t):
    return c4_ganador(t) != 0 or not c4_movs(t)


def c4_mostrar(t):
    simbolo = {0: "·", 1: "X", 2: "O"}
    for f in reversed(range(FILAS)):
        print(" ".join(simbolo[c4_celda(t, f, c)] for c in range(COLS)))
    print(" ".join(str(c) for c in range(COLS)))


t = c4_vacio()
for c in [3, 3, 4, 2, 5]:
    t = c4_jugar(t, c)
c4_mostrar(t)

### 5.1 Una función de evaluación

La función `evaluar` puntúa la posición **desde el punto de vista del jugador 1** (`X`, MAX). Revisa cada ventana de 4 celdas: si solo contiene fichas de un jugador, suma (o resta) más puntos cuantas más fichas tenga. Además premia las fichas en la columna central, que participan en más líneas.

In [ ]:
PESOS = {2: 2, 3: 10}                         # puntos por ventana con 2 o 3 fichas propias
GANA = 10_000


def evaluar(t, pesos=PESOS, centro=3):
    g = c4_ganador(t)
    if g:
        return GANA if g == 1 else -GANA
    puntos = 0
    for v in c4_ventanas(t):
        x, o = v.count(1), v.count(2)
        if x and not o:
            puntos += pesos.get(x, 0)
        elif o and not x:
            puntos -= pesos.get(o, 0)
    col = t[3]
    puntos += centro * (col.count(1) - col.count(2))
    return puntos


print("Evaluación de la posición anterior:", evaluar(t))

### 5.2 Alfa-beta con profundidad limitada

Es la misma poda de la sección 4, con dos cambios: un parámetro `profundidad` que se reduce en cada nivel y, al llegar a 0, `evaluar` en lugar de la utilidad. Las victorias se ajustan con la profundidad para que el agente prefiera ganar **pronto** y perder **tarde**. Las columnas se exploran del centro hacia los bordes.

In [ ]:
ORDEN_C4 = [3, 2, 4, 1, 5, 0, 6]
contador = {"nodos": 0}


def c4_alfabeta(t, profundidad, alfa=-math.inf, beta=math.inf, evaluacion=evaluar, podar=True):
    contador["nodos"] += 1
    g = c4_ganador(t)
    if g:
        return (GANA + profundidad) if g == 1 else -(GANA + profundidad)
    movs = [c for c in ORDEN_C4 if len(t[c]) < FILAS]
    if not movs:
        return 0
    if profundidad == 0:
        return evaluacion(t)
    if c4_turno(t) == 1:
        v = -math.inf
        for c in movs:
            v = max(v, c4_alfabeta(c4_jugar(t, c), profundidad - 1, alfa, beta, evaluacion, podar))
            alfa = max(alfa, v)
            if podar and alfa >= beta:
                break
        return v
    else:
        v = math.inf
        for c in movs:
            v = min(v, c4_alfabeta(c4_jugar(t, c), profundidad - 1, alfa, beta, evaluacion, podar))
            beta = min(beta, v)
            if podar and alfa >= beta:
                break
        return v


def c4_decidir(t, profundidad, evaluacion=evaluar):
    max_j = c4_turno(t) == 1
    mejor, mejor_v = None, None
    for c in [c for c in ORDEN_C4 if len(t[c]) < FILAS]:
        v = c4_alfabeta(c4_jugar(t, c), profundidad - 1, evaluacion=evaluacion)
        if mejor_v is None or (v > mejor_v if max_j else v < mejor_v):
            mejor, mejor_v = c, v
    return mejor


print("Columna elegida con profundidad 4:", c4_decidir(t, 4))

### ✏️ Ejercicio 3. El costo de mirar más lejos

1. Ejecuta la celda. ¿Cómo crecen los nodos visitados con la profundidad, con y sin poda? Calcula el cociente entre una profundidad y la siguiente: es una estimación del factor de ramificación efectivo.
2. ¿Qué porcentaje de nodos ahorra la poda en la profundidad 5?
3. Si tu computador visita unos 100.000 nodos por segundo en Python, ¿hasta qué profundidad podrías buscar en 10 segundos con poda? ¿Y sin poda?

In [ ]:
print(f"{'prof.':>5}{'minimax':>12}{'alfa-beta':>12}{'ahorro':>9}")
for p in range(1, 6):
    contador["nodos"] = 0
    v1 = c4_alfabeta(c4_vacio(), p, podar=False)
    sin_poda = contador["nodos"]
    contador["nodos"] = 0
    v2 = c4_alfabeta(c4_vacio(), p, podar=True)
    con_poda = contador["nodos"]
    assert v1 == v2                             # la poda no cambia el valor
    print(f"{p:>5}{sin_poda:>12,}{con_poda:>12,}{100 * (1 - con_poda / sin_poda):>8.1f}%")

*Tus respuestas al ejercicio 3:*

## 6. Un torneo de Conecta 4

Enfrentamos agentes de distinta profundidad. Para que las partidas no sean siempre idénticas, los dos primeros movimientos de cada partida se eligen al azar.

In [ ]:
def c4_partida(agente1, agente2, aperturas=2, rng=None):
    t = c4_vacio()
    for _ in range(aperturas):
        t = c4_jugar(t, rng.choice(c4_movs(t)))
    while not c4_terminal(t):
        agente = agente1 if c4_turno(t) == 1 else agente2
        t = c4_jugar(t, agente(t))
    return c4_ganador(t)


def torneo(agente_a, agente_b, partidas=10, semilla=0):
    rng = random.Random(semilla)
    marcador = {"A": 0, "B": 0, "empate": 0}
    for k in range(partidas):
        if k % 2 == 0:
            g = c4_partida(agente_a, agente_b, rng=rng)
            marcador["A" if g == 1 else "B" if g == 2 else "empate"] += 1
        else:
            g = c4_partida(agente_b, agente_a, rng=rng)
            marcador["A" if g == 2 else "B" if g == 1 else "empate"] += 1
    return marcador


rng_global = random.Random(5)
al_azar = lambda t: rng_global.choice(c4_movs(t))
prof = lambda p: (lambda t: c4_decidir(t, p))

t0 = time.perf_counter()
print("Profundidad 2 contra azar:        ", torneo(prof(2), al_azar))
print("Profundidad 4 contra profundidad 1:", torneo(prof(4), prof(1)))
print("Profundidad 4 contra profundidad 2:", torneo(prof(4), prof(2)))
print(f"Tiempo total: {time.perf_counter() - t0:.0f} s")

### ✏️ Ejercicio 4. Diseña tu función de evaluación

La calidad de un agente con profundidad limitada depende tanto de cuánto mira como de **cómo evalúa** (sección 4.4).

1. Escribe una función `mi_evaluacion(t)` que cambie los pesos de `PESOS`, el peso del centro o que añada otro rasgo (por ejemplo, penalizar las ventanas de 3 fichas del rival que tienen la cuarta celda jugable de inmediato).
2. Enfrenta un agente de profundidad 3 con tu evaluación contra uno de profundidad 3 con la evaluación original en 10 partidas. ¿Ganó tu versión?
3. Explica por qué una evaluación con información de más calidad puede compensar uno o dos niveles de profundidad.

In [ ]:
def mi_evaluacion(t):
    # TODO: modifica los pesos o añade rasgos.
    return evaluar(t, pesos={2: 3, 3: 15}, centro=4)


mio = lambda t: c4_decidir(t, 3, evaluacion=mi_evaluacion)
original = lambda t: c4_decidir(t, 3)
print("Mi evaluación (A) contra la original (B):", torneo(mio, original, partidas=10, semilla=3))

*Tus respuestas al ejercicio 4:*

## 7. Juegos con azar: expectiminimax

En los juegos con dados, como el *backgammon*, el árbol tiene además **nodos de azar**. Su valor es el **promedio ponderado** de los valores de sus hijos según la probabilidad de cada resultado: es el algoritmo **expectiminimax** (sección 4.6.2).

Un ejemplo mínimo: MAX elige entre dos jugadas, **A** y **B**. Después se lanza una moneda y el resultado depende de lo que salga:

| Jugada | Cara (p = 0,9) | Sello (p = 0,1) |
|---|:-:|:-:|
| A | 2 | 3 |
| B | 1 | 4 |

In [ ]:
arbol = {
    "A": [(0.9, 2), (0.1, 3)],
    "B": [(0.9, 1), (0.1, 4)],
}


def expectiminimax(arbol, f=lambda x: x):
    # Valor esperado de cada jugada de MAX, aplicando f a los valores de las hojas.
    valores = {j: sum(p * f(v) for p, v in hijos) for j, hijos in arbol.items()}
    return max(valores, key=valores.get), valores


# Una transformación que conserva el orden: 1 < 2 < 3 < 4  ->  1 < 20 < 30 < 400
transformar = {1: 1, 2: 20, 3: 30, 4: 400}
print("Con los valores originales:  ", expectiminimax(arbol))
print("Con los valores transformados:", expectiminimax(arbol, f=transformar.get))

### ✏️ Ejercicio 5. La escala de la evaluación

1. La transformación de la celda anterior cambia los valores 1, 2, 3 y 4 por 1, 20, 30 y 400: conserva su orden. En un juego **sin azar**, ¿cambiaría la decisión de minimax si se aplicara a todas las hojas? Justifícalo.
2. ¿Por qué en el ejemplo anterior la decisión **sí** cambia? Calcula a mano los dos valores esperados.
3. Russell y Norvig (2004, cap. 6) concluyen que en los juegos con azar la función de evaluación debe ser una transformación lineal positiva de la probabilidad de ganar. Explica esa conclusión con tus palabras a partir de este ejemplo.

*Tus respuestas al ejercicio 5:*

## 8. Cierre

Completa esta tabla con lo que observaste:

| Técnica | ¿Qué problema resuelve? | ¿Cambia el valor del juego? | Costo principal |
|---|---|---|---|
| Minimax | | | |
| Negamax | | | |
| Poda alfa-beta | | | |
| Ordenación de movimientos | | | |
| Profundidad limitada + función de evaluación | | | |
| Expectiminimax | | | |

Lleva el cuaderno ejecutado a la sesión sincrónica. El agente de Conecta 4 es el punto de partida de la parte B de la **Actividad Evaluativa 2**.

---

**Referencias**

Knuth, D. E., y Moore, R. W. (1975). An analysis of alpha-beta pruning. *Artificial Intelligence, 6*(4), 293-326. https://doi.org/10.1016/0004-3702(75)90019-3

Russell, S. J., y Norvig, P. (2004). *Inteligencia artificial: Un enfoque moderno* (2.ª ed.; J. M. Corchado Rodríguez et al., Trads.). Pearson Educación.

Shannon, C. E. (1950). Programming a computer for playing chess. *Philosophical Magazine, 41*(314), 256-275. https://doi.org/10.1080/14786445008521796